In [10]:
# ============================================================
# NOTEBOOK 16 — GPA_S1 EXCLUSION SENSITIVITY ANALYSIS
# CELL 1 — INITIALISATION, SCOPE, AND GUARDRAILS
# ============================================================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. ANALYSIS STATUS
# ------------------------------------------------------------
ANALYSIS_NAME = "GPA_S1 Exclusion Sensitivity Analysis"
PRIMARY_PIPELINE_STATUS = "LOCKED — DO NOT MODIFY"

print("=" * 72)
print("GPA_S1 EXCLUSION SENSITIVITY ANALYSIS")
print("=" * 72)
print(f"Analysis: {ANALYSIS_NAME}")
print(f"Primary pipeline status: {PRIMARY_PIPELINE_STATUS}")

# ------------------------------------------------------------
# 2. PROJECT PATHS
# ------------------------------------------------------------
# Project directories
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
FIELD_DIR = DATA_DIR / "field"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURE_DIR = RESULTS_DIR / "figures"
TABLE_DIR = RESULTS_DIR / "tables"
REPORT_DIR = RESULTS_DIR / "reports"
# Create output directories
SENSITIVITY_DIR = RESULTS_DIR / "gpa_s1_exclusion_sensitivity"
SENSITIVITY_MODEL_DIR = MODELS_DIR / "gpa_s1_exclusion_sensitivity"

SENSITIVITY_DIR.mkdir(parents=True, exist_ok=True)
SENSITIVITY_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("\nSensitivity output directory:")
print(SENSITIVITY_DIR.resolve())

print("\nSensitivity model directory:")
print(SENSITIVITY_MODEL_DIR.resolve())

# ------------------------------------------------------------
# 3. LOCKED STUDY CONSTANTS
# ------------------------------------------------------------
EXPECTED_TRAIN_N = 468
EXPECTED_TEST_N = 117

LOCKED_FEATURE_COUNT = 49
EXPECTED_SENSITIVITY_FEATURE_COUNT = 48

EXCLUDED_FEATURE = "GPA_S1"

SEEDS = [42, 123, 7]

DECISION_THRESHOLD = 0.50

print("\nLocked study constants:")
print(f"  Training observations: {EXPECTED_TRAIN_N}")
print(f"  Test observations:     {EXPECTED_TEST_N}")
print(f"  Locked feature count:  {LOCKED_FEATURE_COUNT}")
print(f"  Sensitivity features:  {EXPECTED_SENSITIVITY_FEATURE_COUNT}")
print(f"  Excluded predictor:    {EXCLUDED_FEATURE}")
print(f"  Seeds:                 {SEEDS}")
print(f"  Decision threshold:    {DECISION_THRESHOLD:.2f}")

# ------------------------------------------------------------
# 4. ANALYSIS GUARDRAIL
# ------------------------------------------------------------
ANALYSIS_RULES = {
    "primary_pipeline_modified": False,
    "feature_reselection_permitted": False,
    "hyperparameter_retuning_permitted": False,
    "test_set_used_for_tuning": False,
    "excluded_feature": EXCLUDED_FEATURE,
    "comparison_type": "post-hoc sensitivity analysis"
}

guardrail_path = SENSITIVITY_DIR / "sensitivity_analysis_guardrails.json"

with open(guardrail_path, "w", encoding="utf-8") as f:
    json.dump(ANALYSIS_RULES, f, indent=4)

print("\nAnalysis guardrails:")
for key, value in ANALYSIS_RULES.items():
    print(f"  {key}: {value}")

print("\n✓ Notebook 16 initialised")
print("✓ Primary SP-XGBoost workflow remains locked")
print("✓ No feature reselection permitted")
print("✓ No hyperparameter retuning permitted")
print("✓ GPA_S1 will be excluded only in the cloned sensitivity branch")
print(f"✓ Guardrail artifact saved: {guardrail_path}")
print("=" * 72)

GPA_S1 EXCLUSION SENSITIVITY ANALYSIS
Analysis: GPA_S1 Exclusion Sensitivity Analysis
Primary pipeline status: LOCKED — DO NOT MODIFY

Sensitivity output directory:
C:\Users\HP\Documents\SP-XGBOOST\results\gpa_s1_exclusion_sensitivity

Sensitivity model directory:
C:\Users\HP\Documents\SP-XGBOOST\models\gpa_s1_exclusion_sensitivity

Locked study constants:
  Training observations: 468
  Test observations:     117
  Locked feature count:  49
  Sensitivity features:  48
  Excluded predictor:    GPA_S1
  Seeds:                 [42, 123, 7]
  Decision threshold:    0.50

Analysis guardrails:
  primary_pipeline_modified: False
  feature_reselection_permitted: False
  hyperparameter_retuning_permitted: False
  test_set_used_for_tuning: False
  excluded_feature: GPA_S1
  comparison_type: post-hoc sensitivity analysis

✓ Notebook 16 initialised
✓ Primary SP-XGBoost workflow remains locked
✓ No feature reselection permitted
✓ No hyperparameter retuning permitted
✓ GPA_S1 will be excluded only i

In [11]:
# ============================================================
# NOTEBOOK 16 — GPA_S1 EXCLUSION SENSITIVITY ANALYSIS
# CELL 2 — IDENTIFY AND VALIDATE LOCKED 49-FEATURE PROVENANCE
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 76)
print("CELL 2B — LOCKED FEATURE-SPACE PROVENANCE VALIDATION")
print("=" * 76)

# ------------------------------------------------------------
# 1. AUTHORITATIVE EXISTING ARTIFACT PATHS
# ------------------------------------------------------------
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"

train_model_path = PROCESSED_DIR / "train_binary_model.csv"
test_model_path = PROCESSED_DIR / "test_binary_model.csv"

selected_features_path = RESULTS_DIR / "shap_selected_features_binary.csv"
feature_provenance_path = RESULTS_DIR / "shap_feature_provenance_binary.csv"

best_params_path = RESULTS_DIR / "sp_xgboost_best_params.csv"
locked_model_path = MODELS_DIR / "sp_xgboost_binary.pkl"

required_paths = {
    "train_binary_model": train_model_path,
    "test_binary_model": test_model_path,
    "SHAP selected features": selected_features_path,
    "SP-XGBoost best parameters": best_params_path,
    "locked SP-XGBoost model": locked_model_path,
}

print("\nRequired artifact check:")

for label, path in required_paths.items():
    status = "✓" if path.exists() else "✗"
    print(f"  {status} {label}: {path}")

missing_required = [
    label
    for label, path in required_paths.items()
    if not path.exists()
]

assert not missing_required, (
    "Missing required locked artifacts: "
    + ", ".join(missing_required)
)

# ------------------------------------------------------------
# 2. LOAD MODEL-READY BINARY DATA
# ------------------------------------------------------------
train_model_df = pd.read_csv(train_model_path)
test_model_df = pd.read_csv(test_model_path)

print("\nModel-ready dataset shapes:")
print(f"  train_binary_model.csv: {train_model_df.shape}")
print(f"  test_binary_model.csv:  {test_model_df.shape}")

print("\nTraining columns:")
print(list(train_model_df.columns))

print("\nTest columns:")
print(list(test_model_df.columns))

# ------------------------------------------------------------
# 3. IDENTIFY BINARY TARGET COLUMN
# ------------------------------------------------------------
possible_targets = [
    "RISK_BINARY",
    "Risk_Binary",
    "risk_binary",
    "target",
    "Target",
    "y"
]

target_candidates = [
    col for col in possible_targets
    if col in train_model_df.columns
]

if len(target_candidates) == 1:
    TARGET_COL = target_candidates[0]

elif len(target_candidates) == 0:
    TARGET_COL = None

else:
    raise ValueError(
        f"Multiple possible target columns detected: {target_candidates}"
    )

print("\nTarget-column inspection:")
print(f"  Detected target column: {TARGET_COL}")

# ------------------------------------------------------------
# 4. LOAD AUTHORITATIVE SHAP-SELECTED FEATURE ARTIFACT
# ------------------------------------------------------------
selected_df = pd.read_csv(selected_features_path)

print("\nSHAP-selected feature artifact:")
print(f"  Shape: {selected_df.shape}")
print(f"  Columns: {list(selected_df.columns)}")

print("\nFirst 10 rows:")
print(selected_df.head(10).to_string(index=False))

# ------------------------------------------------------------
# 5. DETECT FEATURE-NAME COLUMN SAFELY
# ------------------------------------------------------------
feature_column_candidates = [
    "feature",
    "Feature",
    "feature_name",
    "Feature_Name",
    "selected_feature",
    "Selected_Feature",
    "Predictor",
    "predictor"
]

matched_feature_columns = [
    col for col in feature_column_candidates
    if col in selected_df.columns
]

if len(matched_feature_columns) == 1:
    FEATURE_NAME_COL = matched_feature_columns[0]

elif len(matched_feature_columns) > 1:
    raise ValueError(
        "Multiple possible feature-name columns detected: "
        f"{matched_feature_columns}"
    )

else:
    # Safe fallback only when the artifact contains exactly one text column
    object_cols = selected_df.select_dtypes(
        include=["object", "string"]
    ).columns.tolist()

    if len(object_cols) == 1:
        FEATURE_NAME_COL = object_cols[0]
    else:
        FEATURE_NAME_COL = None

print("\nFeature-name column inspection:")
print(f"  Detected feature-name column: {FEATURE_NAME_COL}")

# ------------------------------------------------------------
# 6. EXTRACT LOCKED FEATURE LIST IF IDENTIFIED
# ------------------------------------------------------------
if FEATURE_NAME_COL is not None:

    locked_features = (
        selected_df[FEATURE_NAME_COL]
        .dropna()
        .astype(str)
        .tolist()
    )

    print("\nLocked feature-list summary:")
    print(f"  Number of rows/features: {len(locked_features)}")
    print(f"  Unique features:         {len(set(locked_features))}")
    print(f"  GPA_S1 present:          {EXCLUDED_FEATURE in locked_features}")

    if EXCLUDED_FEATURE in locked_features:
        print(
            f"  GPA_S1 position:         "
            f"{locked_features.index(EXCLUDED_FEATURE) + 1}"
        )

else:
    locked_features = None

    print(
        "\n⚠ Feature-name column could not yet be identified safely."
    )

# ------------------------------------------------------------
# 7. INSPECT MODEL DATA FEATURE COUNTS
# ------------------------------------------------------------
if TARGET_COL is not None:

    train_predictor_cols = [
        col for col in train_model_df.columns
        if col != TARGET_COL
    ]

    test_predictor_cols = [
        col for col in test_model_df.columns
        if col != TARGET_COL
    ]

else:

    train_predictor_cols = list(train_model_df.columns)
    test_predictor_cols = list(test_model_df.columns)

print("\nModel-ready predictor inspection:")
print(f"  Training predictor count: {len(train_predictor_cols)}")
print(f"  Test predictor count:     {len(test_predictor_cols)}")
print(
    f"  Train/test predictor order identical: "
    f"{train_predictor_cols == test_predictor_cols}"
)
print(
    f"  GPA_S1 in model-ready training data: "
    f"{EXCLUDED_FEATURE in train_predictor_cols}"
)

# ------------------------------------------------------------
# 8. COMPARE SHAP LIST AGAINST MODEL-READY DATA
# ------------------------------------------------------------
if locked_features is not None:

    missing_from_train = [
        f for f in locked_features
        if f not in train_predictor_cols
    ]

    missing_from_test = [
        f for f in locked_features
        if f not in test_predictor_cols
    ]

    extra_train_features = [
        f for f in train_predictor_cols
        if f not in locked_features
    ]

    print("\nFeature provenance comparison:")
    print(
        f"  SHAP features missing from train: "
        f"{len(missing_from_train)}"
    )
    print(
        f"  SHAP features missing from test:  "
        f"{len(missing_from_test)}"
    )
    print(
        f"  Additional model-ready predictors: "
        f"{len(extra_train_features)}"
    )

    if missing_from_train:
        print("\nMissing from training:")
        print(missing_from_train)

    if missing_from_test:
        print("\nMissing from test:")
        print(missing_from_test)

# ------------------------------------------------------------
# 9. SAMPLE-SIZE CHECK
# ------------------------------------------------------------
assert len(train_model_df) == EXPECTED_TRAIN_N, (
    f"Expected {EXPECTED_TRAIN_N} training rows, "
    f"found {len(train_model_df)}"
)

assert len(test_model_df) == EXPECTED_TEST_N, (
    f"Expected {EXPECTED_TEST_N} test rows, "
    f"found {len(test_model_df)}"
)

print("\nSample-size validation:")
print(f"✓ Training observations: {len(train_model_df)}")
print(f"✓ Test observations:     {len(test_model_df)}")

# ------------------------------------------------------------
# 10. DO NOT YET CONSTRUCT THE 48-FEATURE DATA
# ------------------------------------------------------------
print("\n" + "=" * 76)
print("CELL 2B COMPLETE")
print("=" * 76)

print(
    "\nNo features have been removed and no model has been fitted."
)
print(
    "This cell only inspects and validates existing locked artifacts."
)

CELL 2B — LOCKED FEATURE-SPACE PROVENANCE VALIDATION

Required artifact check:
  ✓ train_binary_model: ..\data\processed\train_binary_model.csv
  ✓ test_binary_model: ..\data\processed\test_binary_model.csv
  ✓ SHAP selected features: ..\results\shap_selected_features_binary.csv
  ✓ SP-XGBoost best parameters: ..\results\sp_xgboost_best_params.csv
  ✓ locked SP-XGBoost model: ..\models\sp_xgboost_binary.pkl

Model-ready dataset shapes:
  train_binary_model.csv: (468, 62)
  test_binary_model.csv:  (117, 62)

Training columns:
['GPA_S1', 'CA_AVG', 'EXAM_AVG', 'CLIN_AVG', 'LAB_AVG', 'ATT_RATE', 'ASSIGN_LATE', 'Age_group', 'SES', 'Financial_diff', 'Employment_hrs', 'Study_hrs_day', 'Sleep_hrs', 'Self_risk_percep', 'Reviews notes within 24h', 'Understands content pre-exam', 'Seeks help when stuck', 'Uses library regularly', 'Completes readings', 'Takes organised notes', 'Concentration in self-study', 'Participates in class', 'Clinical takes study time', 'Prepared for clinical assess', 'Rota

In [12]:
# ============================================================
# NOTEBOOK 16 — GPA_S1 EXCLUSION SENSITIVITY ANALYSIS
# CELL 3 — RECONSTRUCT LOCKED 49-FEATURE SPACE
#          AND CREATE 48-FEATURE GPA_S1-EXCLUDED CLONE
# ============================================================

import pandas as pd
import numpy as np

print("=" * 78)
print("CELL 3 — LOCKED 49-FEATURE RECONSTRUCTION AND GPA_S1 EXCLUSION")
print("=" * 78)

# ------------------------------------------------------------
# 1. EXTRACT TARGETS FROM MODEL-READY DATA
# ------------------------------------------------------------
y_train = train_model_df[TARGET_COL].copy()
y_test = test_model_df[TARGET_COL].copy()

print("\nTarget shapes:")
print(f"  y_train: {y_train.shape}")
print(f"  y_test:  {y_test.shape}")

# ------------------------------------------------------------
# 2. RECONSTRUCT EXACT LOCKED 49-FEATURE MATRICES
# ------------------------------------------------------------
X_train_49 = train_model_df[locked_features].copy()
X_test_49 = test_model_df[locked_features].copy()

print("\nLocked 49-feature matrices:")
print(f"  X_train_49: {X_train_49.shape}")
print(f"  X_test_49:  {X_test_49.shape}")

# ------------------------------------------------------------
# 3. VALIDATE LOCKED FEATURE ORDER
# ------------------------------------------------------------
assert list(X_train_49.columns) == locked_features, (
    "Training feature order does not match authoritative locked feature list."
)

assert list(X_test_49.columns) == locked_features, (
    "Test feature order does not match authoritative locked feature list."
)

assert X_train_49.shape == (
    EXPECTED_TRAIN_N,
    LOCKED_FEATURE_COUNT
), (
    f"Unexpected X_train_49 shape: {X_train_49.shape}"
)

assert X_test_49.shape == (
    EXPECTED_TEST_N,
    LOCKED_FEATURE_COUNT
), (
    f"Unexpected X_test_49 shape: {X_test_49.shape}"
)

print("\n✓ Exact locked 49-feature space reconstructed")

# ------------------------------------------------------------
# 4. CONFIRM GPA_S1 BEFORE EXCLUSION
# ------------------------------------------------------------
assert EXCLUDED_FEATURE in X_train_49.columns
assert EXCLUDED_FEATURE in X_test_49.columns

print(f"✓ {EXCLUDED_FEATURE} confirmed before exclusion")
print(
    f"✓ {EXCLUDED_FEATURE} locked rank/position: "
    f"{locked_features.index(EXCLUDED_FEATURE) + 1}"
)

# ------------------------------------------------------------
# 5. CREATE 48-FEATURE SENSITIVITY FEATURE LIST
# ------------------------------------------------------------
sensitivity_features = [
    feature
    for feature in locked_features
    if feature != EXCLUDED_FEATURE
]

assert len(sensitivity_features) == EXPECTED_SENSITIVITY_FEATURE_COUNT, (
    f"Expected {EXPECTED_SENSITIVITY_FEATURE_COUNT} predictors after exclusion, "
    f"found {len(sensitivity_features)}"
)

assert EXCLUDED_FEATURE not in sensitivity_features

print("\nSensitivity feature-space construction:")
print(f"  Locked features before exclusion: {len(locked_features)}")
print(f"  Removed features:                 1")
print(f"  Removed predictor:                {EXCLUDED_FEATURE}")
print(f"  Remaining predictors:             {len(sensitivity_features)}")

# ------------------------------------------------------------
# 6. CREATE CLONED 48-FEATURE MATRICES
# ------------------------------------------------------------
X_train_48 = X_train_49[sensitivity_features].copy()
X_test_48 = X_test_49[sensitivity_features].copy()

print("\nGPA_S1-excluded matrices:")
print(f"  X_train_48: {X_train_48.shape}")
print(f"  X_test_48:  {X_test_48.shape}")

# ------------------------------------------------------------
# 7. STRICT DIFFERENCE AUDIT
# ------------------------------------------------------------
removed_from_train = [
    col for col in X_train_49.columns
    if col not in X_train_48.columns
]

removed_from_test = [
    col for col in X_test_49.columns
    if col not in X_test_48.columns
]

added_to_train = [
    col for col in X_train_48.columns
    if col not in X_train_49.columns
]

added_to_test = [
    col for col in X_test_48.columns
    if col not in X_test_49.columns
]

assert removed_from_train == [EXCLUDED_FEATURE], (
    f"Unexpected training features removed: {removed_from_train}"
)

assert removed_from_test == [EXCLUDED_FEATURE], (
    f"Unexpected test features removed: {removed_from_test}"
)

assert len(added_to_train) == 0, (
    f"Unexpected training features added: {added_to_train}"
)

assert len(added_to_test) == 0, (
    f"Unexpected test features added: {added_to_test}"
)

print("\nStrict difference audit:")
print(f"  Removed from training: {removed_from_train}")
print(f"  Removed from test:     {removed_from_test}")
print(f"  Added to training:     {added_to_train}")
print(f"  Added to test:         {added_to_test}")

# ------------------------------------------------------------
# 8. VERIFY ALL REMAINING VALUES ARE IDENTICAL
# ------------------------------------------------------------
train_values_identical = (
    X_train_49[sensitivity_features]
    .equals(X_train_48)
)

test_values_identical = (
    X_test_49[sensitivity_features]
    .equals(X_test_48)
)

assert train_values_identical, (
    "Training values changed unexpectedly during cloning."
)

assert test_values_identical, (
    "Test values changed unexpectedly during cloning."
)

print("\nValue-preservation audit:")
print(f"  Training remaining values identical: {train_values_identical}")
print(f"  Test remaining values identical:     {test_values_identical}")

# ------------------------------------------------------------
# 9. CHECK MISSING VALUES
# ------------------------------------------------------------
train_missing_49 = int(X_train_49.isna().sum().sum())
test_missing_49 = int(X_test_49.isna().sum().sum())

train_missing_48 = int(X_train_48.isna().sum().sum())
test_missing_48 = int(X_test_48.isna().sum().sum())

print("\nMissing-value audit:")
print(f"  Locked train 49:      {train_missing_49}")
print(f"  Locked test 49:       {test_missing_49}")
print(f"  Sensitivity train 48: {train_missing_48}")
print(f"  Sensitivity test 48:  {test_missing_48}")

# ------------------------------------------------------------
# 10. TARGET DISTRIBUTION VALIDATION
# ------------------------------------------------------------
train_distribution = y_train.value_counts().sort_index()
test_distribution = y_test.value_counts().sort_index()

print("\nTarget distributions:")
print("Training:")
print(train_distribution)

print("\nTest:")
print(test_distribution)

assert int(train_distribution.loc[0]) == 244
assert int(train_distribution.loc[1]) == 224

assert int(test_distribution.loc[0]) == 61
assert int(test_distribution.loc[1]) == 56

print("\n✓ Expected binary class distributions confirmed")

# ------------------------------------------------------------
# 11. SAVE SENSITIVITY FEATURE MANIFEST ONLY
# ------------------------------------------------------------
sensitivity_manifest = pd.DataFrame({
    "sensitivity_order": range(
        1,
        len(sensitivity_features) + 1
    ),
    "feature_name": sensitivity_features
})

sensitivity_manifest_path = (
    SENSITIVITY_DIR /
    "gpa_s1_excluded_48_feature_manifest.csv"
)

sensitivity_manifest.to_csv(
    sensitivity_manifest_path,
    index=False
)

# Separate exclusion audit
exclusion_audit = pd.DataFrame({
    "analysis": ["GPA_S1 exclusion sensitivity"],
    "locked_feature_count": [len(locked_features)],
    "excluded_feature": [EXCLUDED_FEATURE],
    "excluded_count": [1],
    "sensitivity_feature_count": [len(sensitivity_features)],
    "feature_reselection": [False],
    "hyperparameter_retuning": [False]
})

exclusion_audit_path = (
    SENSITIVITY_DIR /
    "gpa_s1_exclusion_feature_audit.csv"
)

exclusion_audit.to_csv(
    exclusion_audit_path,
    index=False
)

# ------------------------------------------------------------
# 12. FINAL VALIDATION
# ------------------------------------------------------------
print("\n" + "=" * 78)
print("CELL 3 VALIDATION SUMMARY")
print("=" * 78)

print(f"✓ Locked training matrix preserved: {X_train_49.shape}")
print(f"✓ Locked test matrix preserved:     {X_test_49.shape}")
print(f"✓ Sensitivity training matrix:      {X_train_48.shape}")
print(f"✓ Sensitivity test matrix:          {X_test_48.shape}")
print(f"✓ Only {EXCLUDED_FEATURE} was removed")
print("✓ No predictor was added")
print("✓ No feature reselection performed")
print("✓ No hyperparameter optimisation performed")
print("✓ Original 49-feature objects remain available")
print("✓ Test cohort remains unchanged")
print(f"✓ 48-feature manifest saved: {sensitivity_manifest_path}")
print(f"✓ Exclusion audit saved:     {exclusion_audit_path}")

print("=" * 78)

CELL 3 — LOCKED 49-FEATURE RECONSTRUCTION AND GPA_S1 EXCLUSION

Target shapes:
  y_train: (468,)
  y_test:  (117,)

Locked 49-feature matrices:
  X_train_49: (468, 49)
  X_test_49:  (117, 49)

✓ Exact locked 49-feature space reconstructed
✓ GPA_S1 confirmed before exclusion
✓ GPA_S1 locked rank/position: 1

Sensitivity feature-space construction:
  Locked features before exclusion: 49
  Removed features:                 1
  Removed predictor:                GPA_S1
  Remaining predictors:             48

GPA_S1-excluded matrices:
  X_train_48: (468, 48)
  X_test_48:  (117, 48)

Strict difference audit:
  Removed from training: ['GPA_S1']
  Removed from test:     ['GPA_S1']
  Added to training:     []
  Added to test:         []

Value-preservation audit:
  Training remaining values identical: True
  Test remaining values identical:     True

Missing-value audit:
  Locked train 49:      0
  Locked test 49:       0
  Sensitivity train 48: 0
  Sensitivity test 48:  0

Target distributions:

In [13]:
# ============================================================
# NOTEBOOK 16 — GPA_S1 EXCLUSION SENSITIVITY ANALYSIS
# CELL 4 — LOAD AND VALIDATE LOCKED SP-XGBOOST HYPERPARAMETERS
# ============================================================

import pandas as pd
import numpy as np

print("=" * 78)
print("CELL 4 — LOCKED SP-XGBOOST HYPERPARAMETER VALIDATION")
print("=" * 78)

# ------------------------------------------------------------
# 1. LOAD LOCKED PARAMETER ARTIFACT
# ------------------------------------------------------------
best_params_df = pd.read_csv(best_params_path)

print("\nLocked parameter artifact:")
print(f"  Path:  {best_params_path}")
print(f"  Shape: {best_params_df.shape}")
print(f"  Columns: {list(best_params_df.columns)}")

print("\nRaw contents:")
print(best_params_df.to_string(index=False))

# ------------------------------------------------------------
# 2. DETECT PARAMETER STORAGE FORMAT
# ------------------------------------------------------------
# Supported formats:
# A) two columns such as Parameter / Value
# B) one-row wide table with parameter names as columns

locked_params = {}

if best_params_df.shape[1] == 2:
    col1, col2 = best_params_df.columns

    # Assume first column contains names and second contains values
    locked_params = dict(
        zip(
            best_params_df[col1].astype(str),
            best_params_df[col2]
        )
    )

elif best_params_df.shape[0] == 1:
    locked_params = best_params_df.iloc[0].to_dict()

else:
    raise ValueError(
        "Unable to safely interpret sp_xgboost_best_params.csv.\n"
        f"Shape={best_params_df.shape}, "
        f"Columns={list(best_params_df.columns)}"
    )

print("\nDetected parameter dictionary:")
for key, value in locked_params.items():
    print(f"  {key}: {value}")

# ------------------------------------------------------------
# 3. EXPECTED LOCKED NOTEBOOK 7 PARAMETERS
# ------------------------------------------------------------
expected_locked_params = {
    "n_estimators": 100,
    "max_depth": 3,
    "learning_rate": 0.04133713629478509,
    "min_child_weight": 1,
    "subsample": 0.8493145117240506,
    "colsample_bytree": 0.8332908615865308,
    "gamma": 4.101525122344799,
    "reg_alpha": 0.004811499595716167,
    "reg_lambda": 0.012132398817719065,
}

# ------------------------------------------------------------
# 4. TYPE-NORMALISATION HELPER
# ------------------------------------------------------------
def normalise_param_value(name, value):
    """
    Convert CSV-loaded parameter values into XGBoost-compatible types.
    """

    if isinstance(value, str):
        value = value.strip()

    if name in {"n_estimators", "max_depth", "min_child_weight"}:
        return int(float(value))

    return float(value)

normalised_locked_params = {
    name: normalise_param_value(name, value)
    for name, value in locked_params.items()
    if name in expected_locked_params
}

# ------------------------------------------------------------
# 5. CHECK FOR REQUIRED PARAMETERS
# ------------------------------------------------------------
missing_params = [
    name
    for name in expected_locked_params
    if name not in normalised_locked_params
]

assert not missing_params, (
    "Required locked parameters missing from artifact: "
    + ", ".join(missing_params)
)

print("\nRequired parameter presence:")
for name in expected_locked_params:
    print(f"  ✓ {name}")

# ------------------------------------------------------------
# 6. NUMERICAL PROVENANCE CHECK
# ------------------------------------------------------------
comparison_rows = []

for name, expected_value in expected_locked_params.items():

    artifact_value = normalised_locked_params[name]

    if isinstance(expected_value, int):
        match = artifact_value == expected_value
        abs_difference = abs(artifact_value - expected_value)

    else:
        match = np.isclose(
            artifact_value,
            expected_value,
            rtol=1e-12,
            atol=1e-15
        )
        abs_difference = abs(
            artifact_value - expected_value
        )

    comparison_rows.append({
        "parameter": name,
        "artifact_value": artifact_value,
        "expected_locked_value": expected_value,
        "absolute_difference": abs_difference,
        "match": match
    })

params_validation_df = pd.DataFrame(comparison_rows)

print("\nLocked parameter comparison:")
print(
    params_validation_df.to_string(
        index=False
    )
)

assert params_validation_df["match"].all(), (
    "One or more hyperparameters do not match "
    "the locked Notebook 7 specification."
)

# ------------------------------------------------------------
# 7. DEFINE FINAL LOCKED PARAMETER SET FOR NOTEBOOK 16
# ------------------------------------------------------------
SP_LOCKED_PARAMS = {
    name: normalised_locked_params[name]
    for name in expected_locked_params
}

# Fixed model settings that are NOT being tuned
SP_FIXED_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": None,   # assigned separately for each seed
    "n_jobs": -1,
}

print("\nLocked SP-XGBoost tuning parameters:")
for key, value in SP_LOCKED_PARAMS.items():
    print(f"  {key}: {value}")

print("\nFixed sensitivity-analysis model settings:")
for key, value in SP_FIXED_PARAMS.items():
    print(f"  {key}: {value}")

# ------------------------------------------------------------
# 8. SAVE VALIDATION ARTIFACT
# ------------------------------------------------------------
params_validation_path = (
    SENSITIVITY_DIR /
    "locked_sp_xgboost_hyperparameter_validation.csv"
)

params_validation_df.to_csv(
    params_validation_path,
    index=False
)

# ------------------------------------------------------------
# 9. FINAL GUARDRAIL
# ------------------------------------------------------------
assert ANALYSIS_RULES[
    "hyperparameter_retuning_permitted"
] is False

print("\n" + "=" * 78)
print("CELL 4 VALIDATION SUMMARY")
print("=" * 78)

print("✓ Locked Notebook 7 hyperparameters loaded")
print("✓ All required hyperparameters present")
print("✓ Parameter values match locked specification")
print("✓ No hyperparameter optimisation performed")
print("✓ No test-set information used")
print("✓ Seed will be varied only through random_state")
print(f"✓ Validation artifact saved: {params_validation_path}")

print("=" * 78)

CELL 4 — LOCKED SP-XGBOOST HYPERPARAMETER VALIDATION

Locked parameter artifact:
  Path:  ..\results\sp_xgboost_best_params.csv
  Shape: (1, 9)
  Columns: ['n_estimators', 'max_depth', 'learning_rate', 'min_child_weight', 'subsample', 'colsample_bytree', 'gamma', 'reg_alpha', 'reg_lambda']

Raw contents:
 n_estimators  max_depth  learning_rate  min_child_weight  subsample  colsample_bytree    gamma  reg_alpha  reg_lambda
          100          3       0.041337                 1   0.849315          0.833291 4.101525   0.004811    0.012132

Detected parameter dictionary:
  n_estimators: 100.0
  max_depth: 3.0
  learning_rate: 0.041337136294785
  min_child_weight: 1.0
  subsample: 0.8493145117240506
  colsample_bytree: 0.8332908615865308
  gamma: 4.101525122344799
  reg_alpha: 0.0048114995957161
  reg_lambda: 0.012132398817719

Required parameter presence:
  ✓ n_estimators
  ✓ max_depth
  ✓ learning_rate
  ✓ min_child_weight
  ✓ subsample
  ✓ colsample_bytree
  ✓ gamma
  ✓ reg_alpha
  ✓ r

In [14]:
# ============================================================
# NOTEBOOK 16 — GPA_S1 EXCLUSION SENSITIVITY ANALYSIS
# CELL 5 — THREE-SEED 48-FEATURE MODEL TRAINING
#          AND INDEPENDENT-TEST EVALUATION
# ============================================================

from pathlib import Path
import pickle

import numpy as np
import pandas as pd

from xgboost import XGBClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix
)

print("=" * 80)
print("CELL 5 — GPA_S1-EXCLUDED THREE-SEED MODEL EVALUATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. FINAL PRE-FIT GUARDRAILS
# ------------------------------------------------------------
assert X_train_48.shape == (
    EXPECTED_TRAIN_N,
    EXPECTED_SENSITIVITY_FEATURE_COUNT
)

assert X_test_48.shape == (
    EXPECTED_TEST_N,
    EXPECTED_SENSITIVITY_FEATURE_COUNT
)

assert EXCLUDED_FEATURE not in X_train_48.columns
assert EXCLUDED_FEATURE not in X_test_48.columns

assert list(X_train_48.columns) == list(X_test_48.columns)

assert ANALYSIS_RULES["feature_reselection_permitted"] is False
assert ANALYSIS_RULES["hyperparameter_retuning_permitted"] is False
assert ANALYSIS_RULES["test_set_used_for_tuning"] is False

assert len(y_train) == EXPECTED_TRAIN_N
assert len(y_test) == EXPECTED_TEST_N

print("\nPre-fit guardrails:")
print(f"✓ Training matrix: {X_train_48.shape}")
print(f"✓ Test matrix:     {X_test_48.shape}")
print(f"✓ {EXCLUDED_FEATURE} absent")
print("✓ Train/test feature order identical")
print("✓ Feature reselection disabled")
print("✓ Hyperparameter retuning disabled")
print("✓ Test-set tuning disabled")

# ------------------------------------------------------------
# 2. METRIC FUNCTION
# ------------------------------------------------------------
def calculate_binary_metrics(
    y_true,
    y_prob,
    threshold=0.50
):
    """
    Calculate the locked binary evaluation metrics.
    """

    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    metrics = {
        "ROC_AUC": roc_auc_score(
            y_true,
            y_prob
        ),
        "PR_AUC": average_precision_score(
            y_true,
            y_prob
        ),
        "F1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Specificity": specificity,
        "Accuracy": accuracy_score(
            y_true,
            y_pred
        ),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    }

    return metrics, y_pred

# ------------------------------------------------------------
# 3. STORAGE OBJECTS
# ------------------------------------------------------------
seed_results = []
prediction_records = []

sensitivity_models = {}

# ------------------------------------------------------------
# 4. TRAIN ONE MODEL PER LOCKED SEED
# ------------------------------------------------------------
for seed in SEEDS:

    print("\n" + "-" * 80)
    print(f"SEED {seed}")
    print("-" * 80)

    # --------------------------------------------------------
    # Construct model from LOCKED parameters only
    # --------------------------------------------------------
    model_params = {
        **SP_LOCKED_PARAMS,
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "random_state": seed,
        "n_jobs": -1
    }

    model = XGBClassifier(
        **model_params
    )

    # --------------------------------------------------------
    # Fit using TRAINING DATA ONLY
    # --------------------------------------------------------
    model.fit(
        X_train_48,
        y_train
    )

    # --------------------------------------------------------
    # Independent-test probabilities
    # --------------------------------------------------------
    y_prob = model.predict_proba(
        X_test_48
    )[:, 1]

    # --------------------------------------------------------
    # Locked 0.50 threshold
    # --------------------------------------------------------
    metrics, y_pred = calculate_binary_metrics(
        y_true=y_test,
        y_prob=y_prob,
        threshold=DECISION_THRESHOLD
    )

    # --------------------------------------------------------
    # Store metric results
    # --------------------------------------------------------
    result_row = {
        "Analysis": "GPA_S1 Exclusion Sensitivity",
        "Model": "SP-XGBoost_without_GPA_S1",
        "Seed": seed,
        "Train_N": len(X_train_48),
        "Test_N": len(X_test_48),
        "Feature_Count": X_train_48.shape[1],
        "Excluded_Feature": EXCLUDED_FEATURE,
        "Decision_Threshold": DECISION_THRESHOLD,
        **metrics
    }

    seed_results.append(result_row)

    # --------------------------------------------------------
    # Store individual test predictions
    # --------------------------------------------------------
    for test_position, (
        true_label,
        probability,
        prediction
    ) in enumerate(
        zip(
            np.asarray(y_test),
            y_prob,
            y_pred
        )
    ):

        prediction_records.append({
            "Test_Position": test_position,
            "Seed": seed,
            "y_true": int(true_label),
            "y_prob_GPA_S1_excluded": float(
                probability
            ),
            "y_pred_GPA_S1_excluded": int(
                prediction
            )
        })

    # --------------------------------------------------------
    # Save model separately
    # --------------------------------------------------------
    model_path = (
        SENSITIVITY_MODEL_DIR /
        f"sp_xgboost_without_gpa_s1_seed_{seed}.pkl"
    )

    with open(model_path, "wb") as f:
        pickle.dump(model, f)

    sensitivity_models[seed] = model

    # --------------------------------------------------------
    # Display seed result
    # --------------------------------------------------------
    print(f"ROC-AUC:     {metrics['ROC_AUC']:.6f}")
    print(f"PR-AUC:      {metrics['PR_AUC']:.6f}")
    print(f"F1:          {metrics['F1']:.6f}")
    print(f"Precision:   {metrics['Precision']:.6f}")
    print(f"Recall:      {metrics['Recall']:.6f}")
    print(f"Specificity: {metrics['Specificity']:.6f}")
    print(f"Accuracy:    {metrics['Accuracy']:.6f}")

    print(
        "Confusion matrix: "
        f"TN={metrics['TN']}, "
        f"FP={metrics['FP']}, "
        f"FN={metrics['FN']}, "
        f"TP={metrics['TP']}"
    )

    print(f"✓ Model saved: {model_path}")

# ------------------------------------------------------------
# 5. CREATE THREE-SEED RESULTS TABLE
# ------------------------------------------------------------
sensitivity_seed_results = pd.DataFrame(
    seed_results
)

metric_columns = [
    "ROC_AUC",
    "PR_AUC",
    "F1",
    "Precision",
    "Recall",
    "Specificity",
    "Accuracy"
]

print("\n" + "=" * 80)
print("THREE-SEED INDEPENDENT-TEST RESULTS")
print("=" * 80)

display_columns = [
    "Seed",
    *metric_columns,
    "TN",
    "FP",
    "FN",
    "TP"
]

print(
    sensitivity_seed_results[
        display_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ------------------------------------------------------------
# 6. CALCULATE THREE-SEED MEAN ± SAMPLE SD
# ------------------------------------------------------------
summary_rows = []

for metric in metric_columns:

    values = sensitivity_seed_results[
        metric
    ].astype(float)

    summary_rows.append({
        "Metric": metric,
        "Mean": values.mean(),
        "SD": values.std(ddof=1),
        "Min": values.min(),
        "Max": values.max()
    })

sensitivity_summary = pd.DataFrame(
    summary_rows
)

print("\n" + "=" * 80)
print("GPA_S1-EXCLUDED THREE-SEED SUMMARY")
print("=" * 80)

for _, row in sensitivity_summary.iterrows():

    print(
        f"{row['Metric']:<12}: "
        f"{row['Mean']:.4f} ± "
        f"{row['SD']:.4f}"
    )

# ------------------------------------------------------------
# 7. CREATE PREDICTION TABLE
# ------------------------------------------------------------
sensitivity_predictions = pd.DataFrame(
    prediction_records
)

assert len(sensitivity_predictions) == (
    EXPECTED_TEST_N * len(SEEDS)
)

for seed in SEEDS:

    seed_predictions = sensitivity_predictions[
        sensitivity_predictions["Seed"] == seed
    ]

    assert len(seed_predictions) == EXPECTED_TEST_N

    assert np.array_equal(
        seed_predictions["y_true"].to_numpy(),
        np.asarray(y_test)
    )

print(
    "\n✓ Prediction alignment validated for all "
    f"{len(SEEDS)} seeds"
)

# ------------------------------------------------------------
# 8. SAVE RESULTS
# ------------------------------------------------------------
seed_results_path = (
    SENSITIVITY_DIR /
    "gpa_s1_excluded_three_seed_test_results.csv"
)

summary_path = (
    SENSITIVITY_DIR /
    "gpa_s1_excluded_three_seed_summary.csv"
)

predictions_path = (
    SENSITIVITY_DIR /
    "gpa_s1_excluded_test_predictions.csv"
)

sensitivity_seed_results.to_csv(
    seed_results_path,
    index=False
)

sensitivity_summary.to_csv(
    summary_path,
    index=False
)

sensitivity_predictions.to_csv(
    predictions_path,
    index=False
)

# ------------------------------------------------------------
# 9. POST-FIT PROVENANCE VALIDATION
# ------------------------------------------------------------
for seed, model in sensitivity_models.items():

    # XGBoost should report exactly 48 input features
    assert model.n_features_in_ == (
        EXPECTED_SENSITIVITY_FEATURE_COUNT
    ), (
        f"Seed {seed}: unexpected model feature count "
        f"{model.n_features_in_}"
    )

    if hasattr(model, "feature_names_in_"):
        assert list(
            model.feature_names_in_
        ) == sensitivity_features

print("\n" + "=" * 80)
print("CELL 5 FINAL VALIDATION")
print("=" * 80)

print(f"✓ {len(SEEDS)} sensitivity models fitted")
print(f"✓ Seeds evaluated: {SEEDS}")
print(f"✓ Each model used exactly {X_train_48.shape[1]} predictors")
print(f"✓ {EXCLUDED_FEATURE} excluded from every model")
print("✓ Same 468 training students retained")
print("✓ Same 117 independent-test students retained")
print(f"✓ Same decision threshold retained: {DECISION_THRESHOLD:.2f}")
print("✓ Locked SP-XGBoost hyperparameters retained")
print("✓ No feature reselection performed")
print("✓ No hyperparameter retuning performed")
print("✓ Individual test probabilities preserved")
print("✓ Original locked 49-feature pipeline remains unchanged")

print("\nSaved artifacts:")
print(f"  ✓ Seed results: {seed_results_path}")
print(f"  ✓ Summary:      {summary_path}")
print(f"  ✓ Predictions:  {predictions_path}")
print(f"  ✓ Models:       {SENSITIVITY_MODEL_DIR}")

print("=" * 80)

CELL 5 — GPA_S1-EXCLUDED THREE-SEED MODEL EVALUATION

Pre-fit guardrails:
✓ Training matrix: (468, 48)
✓ Test matrix:     (117, 48)
✓ GPA_S1 absent
✓ Train/test feature order identical
✓ Feature reselection disabled
✓ Hyperparameter retuning disabled
✓ Test-set tuning disabled

--------------------------------------------------------------------------------
SEED 42
--------------------------------------------------------------------------------
ROC-AUC:     0.523126
PR-AUC:      0.550638
F1:          0.523364
Precision:   0.549020
Recall:      0.500000
Specificity: 0.622951
Accuracy:    0.564103
Confusion matrix: TN=38, FP=23, FN=28, TP=28
✓ Model saved: ..\models\gpa_s1_exclusion_sensitivity\sp_xgboost_without_gpa_s1_seed_42.pkl

--------------------------------------------------------------------------------
SEED 123
--------------------------------------------------------------------------------
ROC-AUC:     0.525761
PR-AUC:      0.549312
F1:          0.500000
Precision:   0.519231


In [15]:
# ============================================================
# NOTEBOOK 16 — GPA_S1 EXCLUSION SENSITIVITY ANALYSIS
# CELL 6 — MATCHED 49-FEATURE VS 48-FEATURE COMPARISON
# ============================================================

from pathlib import Path
import pickle

import numpy as np
import pandas as pd

from xgboost import XGBClassifier

print("=" * 84)
print("CELL 6 — MATCHED 49-FEATURE VS GPA_S1-EXCLUDED 48-FEATURE COMPARISON")
print("=" * 84)

# ------------------------------------------------------------
# 1. FINAL MATCHING GUARDRAILS
# ------------------------------------------------------------
assert X_train_49.shape == (468, 49)
assert X_test_49.shape == (117, 49)

assert X_train_48.shape == (468, 48)
assert X_test_48.shape == (117, 48)

assert EXCLUDED_FEATURE in X_train_49.columns
assert EXCLUDED_FEATURE not in X_train_48.columns

assert list(X_train_49.columns)[1:] == list(X_train_48.columns), (
    "Expected the 48-feature sensitivity space to equal "
    "the locked 49-feature space with GPA_S1 removed."
)

print("\nMatched-design validation:")
print("✓ Same 468 training students")
print("✓ Same 117 independent-test students")
print("✓ Same target labels")
print("✓ Same locked hyperparameters")
print(f"✓ Same seeds: {SEEDS}")
print(f"✓ Same decision threshold: {DECISION_THRESHOLD:.2f}")
print("✓ Only analytical difference = presence/absence of GPA_S1")

# ------------------------------------------------------------
# 2. STORAGE
# ------------------------------------------------------------
reference_results = []
reference_prediction_records = []

matched_49_models = {}

# ------------------------------------------------------------
# 3. RECONSTRUCT MATCHED 49-FEATURE REFERENCE MODELS
# ------------------------------------------------------------
for seed in SEEDS:

    print("\n" + "-" * 84)
    print(f"MATCHED 49-FEATURE REFERENCE — SEED {seed}")
    print("-" * 84)

    model_params = {
        **SP_LOCKED_PARAMS,
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "random_state": seed,
        "n_jobs": -1
    }

    reference_model = XGBClassifier(
        **model_params
    )

    reference_model.fit(
        X_train_49,
        y_train
    )

    reference_prob = reference_model.predict_proba(
        X_test_49
    )[:, 1]

    reference_metrics, reference_pred = calculate_binary_metrics(
        y_true=y_test,
        y_prob=reference_prob,
        threshold=DECISION_THRESHOLD
    )

    reference_results.append({
        "Analysis": "Matched 49-feature reference",
        "Model": "SP-XGBoost_with_GPA_S1",
        "Seed": seed,
        "Train_N": len(X_train_49),
        "Test_N": len(X_test_49),
        "Feature_Count": X_train_49.shape[1],
        "Excluded_Feature": "None",
        "Decision_Threshold": DECISION_THRESHOLD,
        **reference_metrics
    })

    for test_position, (
        true_label,
        probability,
        prediction
    ) in enumerate(
        zip(
            np.asarray(y_test),
            reference_prob,
            reference_pred
        )
    ):

        reference_prediction_records.append({
            "Test_Position": test_position,
            "Seed": seed,
            "y_true": int(true_label),
            "y_prob_49": float(probability),
            "y_pred_49": int(prediction)
        })

    matched_49_models[seed] = reference_model

    model_path = (
        SENSITIVITY_MODEL_DIR /
        f"matched_sp_xgboost_49_features_seed_{seed}.pkl"
    )

    with open(model_path, "wb") as f:
        pickle.dump(reference_model, f)

    print(f"ROC-AUC:     {reference_metrics['ROC_AUC']:.6f}")
    print(f"PR-AUC:      {reference_metrics['PR_AUC']:.6f}")
    print(f"F1:          {reference_metrics['F1']:.6f}")
    print(f"Precision:   {reference_metrics['Precision']:.6f}")
    print(f"Recall:      {reference_metrics['Recall']:.6f}")
    print(f"Specificity: {reference_metrics['Specificity']:.6f}")
    print(f"Accuracy:    {reference_metrics['Accuracy']:.6f}")

    print(
        "Confusion matrix: "
        f"TN={reference_metrics['TN']}, "
        f"FP={reference_metrics['FP']}, "
        f"FN={reference_metrics['FN']}, "
        f"TP={reference_metrics['TP']}"
    )

# ------------------------------------------------------------
# 4. BUILD REFERENCE RESULTS TABLE
# ------------------------------------------------------------
matched_49_results = pd.DataFrame(
    reference_results
)

reference_predictions = pd.DataFrame(
    reference_prediction_records
)

assert len(reference_predictions) == (
    EXPECTED_TEST_N * len(SEEDS)
)

# ------------------------------------------------------------
# 5. MERGE 49-FEATURE AND 48-FEATURE RESULTS
# ------------------------------------------------------------
comparison_rows = []

for seed in SEEDS:

    ref_row = matched_49_results[
        matched_49_results["Seed"] == seed
    ].iloc[0]

    sens_row = sensitivity_seed_results[
        sensitivity_seed_results["Seed"] == seed
    ].iloc[0]

    comparison_row = {
        "Seed": seed,
        "Reference_Features": 49,
        "Sensitivity_Features": 48
    }

    for metric in metric_columns:

        ref_value = float(ref_row[metric])
        sens_value = float(sens_row[metric])

        # Defined as sensitivity minus reference
        delta = sens_value - ref_value

        comparison_row[f"{metric}_49"] = ref_value
        comparison_row[f"{metric}_48"] = sens_value
        comparison_row[f"Delta_{metric}"] = delta

        # Percentage change relative to 49-feature reference
        if ref_value != 0:
            comparison_row[
                f"Percent_Change_{metric}"
            ] = (
                delta / ref_value
            ) * 100
        else:
            comparison_row[
                f"Percent_Change_{metric}"
            ] = np.nan

    comparison_rows.append(
        comparison_row
    )

matched_comparison = pd.DataFrame(
    comparison_rows
)

# ------------------------------------------------------------
# 6. DISPLAY SEED-SPECIFIC COMPARISON
# ------------------------------------------------------------
print("\n" + "=" * 84)
print("SEED-SPECIFIC MATCHED PERFORMANCE COMPARISON")
print("=" * 84)

for seed in SEEDS:

    row = matched_comparison[
        matched_comparison["Seed"] == seed
    ].iloc[0]

    print(f"\nSEED {seed}")

    for metric in metric_columns:

        print(
            f"{metric:<12}: "
            f"49-feature={row[f'{metric}_49']:.4f} | "
            f"48-feature={row[f'{metric}_48']:.4f} | "
            f"Δ={row[f'Delta_{metric}']:+.4f}"
        )

# ------------------------------------------------------------
# 7. CALCULATE THREE-SEED MATCHED SUMMARY
# ------------------------------------------------------------
summary_rows = []

for metric in metric_columns:

    ref_values = matched_comparison[
        f"{metric}_49"
    ].astype(float)

    sens_values = matched_comparison[
        f"{metric}_48"
    ].astype(float)

    deltas = matched_comparison[
        f"Delta_{metric}"
    ].astype(float)

    pct_changes = matched_comparison[
        f"Percent_Change_{metric}"
    ].astype(float)

    summary_rows.append({
        "Metric": metric,

        "Reference_49_Mean": ref_values.mean(),
        "Reference_49_SD": ref_values.std(ddof=1),

        "GPA_S1_Excluded_48_Mean": sens_values.mean(),
        "GPA_S1_Excluded_48_SD": sens_values.std(ddof=1),

        "Mean_Delta_48_minus_49": deltas.mean(),
        "Delta_SD": deltas.std(ddof=1),

        "Mean_Percent_Change": pct_changes.mean()
    })

matched_summary = pd.DataFrame(
    summary_rows
)

print("\n" + "=" * 84)
print("THREE-SEED MATCHED SUMMARY")
print("=" * 84)

for _, row in matched_summary.iterrows():

    print(
        f"{row['Metric']:<12}: "
        f"49-feature "
        f"{row['Reference_49_Mean']:.4f} ± "
        f"{row['Reference_49_SD']:.4f} | "
        f"48-feature "
        f"{row['GPA_S1_Excluded_48_Mean']:.4f} ± "
        f"{row['GPA_S1_Excluded_48_SD']:.4f} | "
        f"Δ={row['Mean_Delta_48_minus_49']:+.4f}"
    )

# ------------------------------------------------------------
# 8. CREATE PAIRED TEST-PREDICTION DATASET
# ------------------------------------------------------------
paired_predictions = reference_predictions.merge(
    sensitivity_predictions,
    on=[
        "Test_Position",
        "Seed",
        "y_true"
    ],
    how="inner",
    validate="one_to_one"
)

assert len(paired_predictions) == (
    EXPECTED_TEST_N * len(SEEDS)
)

print(
    "\n✓ Matched prediction records: "
    f"{len(paired_predictions)}"
)

# ------------------------------------------------------------
# 9. VERIFY IDENTICAL TEST LABEL ORDER
# ------------------------------------------------------------
for seed in SEEDS:

    paired_seed = paired_predictions[
        paired_predictions["Seed"] == seed
    ].sort_values(
        "Test_Position"
    )

    assert len(paired_seed) == EXPECTED_TEST_N

    assert np.array_equal(
        paired_seed["y_true"].to_numpy(),
        np.asarray(y_test)
    )

print("✓ Test-label alignment verified for every seed")

# ------------------------------------------------------------
# 10. OPTIONAL NOTEBOOK 12 PROVENANCE CHECK
# ------------------------------------------------------------
# These values are known from the locked Notebook 12 robustness analysis.
# This is NOT used to alter results. It is only a provenance check.

notebook12_expected = {
    42: {
        "ROC_AUC": 0.8088,
        "PR_AUC": 0.7722,
        "F1": 0.7170,
        "Precision": 0.7600,
        "Recall": 0.6786,
        "Specificity": 0.8033,
        "Accuracy": 0.7436
    },
    123: {
        "ROC_AUC": 0.8121,
        "PR_AUC": 0.7779,
        "F1": 0.6600,
        "Precision": 0.7500,
        "Recall": 0.5893,
        "Specificity": 0.8197,
        "Accuracy": 0.7094
    },
    7: {
        "ROC_AUC": 0.8129,
        "PR_AUC": 0.7754,
        "F1": 0.7238,
        "Precision": 0.7755,
        "Recall": 0.6786,
        "Specificity": 0.8197,
        "Accuracy": 0.7521
    }
}

print("\n" + "=" * 84)
print("NOTEBOOK 12 RECONSTRUCTION CHECK")
print("=" * 84)

for seed in SEEDS:

    current_row = matched_49_results[
        matched_49_results["Seed"] == seed
    ].iloc[0]

    print(f"\nSeed {seed}:")

    for metric in metric_columns:

        current_value = float(
            current_row[metric]
        )

        expected_value = notebook12_expected[
            seed
        ][metric]

        difference = (
            current_value - expected_value
        )

        print(
            f"  {metric:<12}: "
            f"current={current_value:.6f} | "
            f"Notebook12≈{expected_value:.4f} | "
            f"diff={difference:+.6f}"
        )

# ------------------------------------------------------------
# 11. SAVE ARTIFACTS
# ------------------------------------------------------------
reference_results_path = (
    SENSITIVITY_DIR /
    "matched_49_feature_three_seed_reference_results.csv"
)

comparison_path = (
    SENSITIVITY_DIR /
    "matched_49_vs_48_gpa_s1_exclusion_comparison.csv"
)

summary_path = (
    SENSITIVITY_DIR /
    "matched_49_vs_48_gpa_s1_exclusion_summary.csv"
)

paired_predictions_path = (
    SENSITIVITY_DIR /
    "matched_49_vs_48_test_predictions.csv"
)

matched_49_results.to_csv(
    reference_results_path,
    index=False
)

matched_comparison.to_csv(
    comparison_path,
    index=False
)

matched_summary.to_csv(
    summary_path,
    index=False
)

paired_predictions.to_csv(
    paired_predictions_path,
    index=False
)

# ------------------------------------------------------------
# 12. FINAL VALIDATION
# ------------------------------------------------------------
print("\n" + "=" * 84)
print("CELL 6 FINAL VALIDATION")
print("=" * 84)

print("✓ 49-feature reference reconstructed independently")
print("✓ 48-feature sensitivity results retained")
print("✓ Same train/test students used in both branches")
print("✓ Same seeds used in both branches")
print("✓ Same locked hyperparameters used in both branches")
print("✓ Same decision threshold used in both branches")
print("✓ Only GPA_S1 differs between model inputs")
print("✓ Seed-specific absolute changes calculated")
print("✓ Three-seed mean changes calculated")
print("✓ Individual paired probabilities preserved")
print("✓ No significance test performed across the three seeds")
print("✓ Primary locked SP-XGBoost pipeline remains unchanged")

print("\nSaved artifacts:")
print(f"  ✓ Reference results: {reference_results_path}")
print(f"  ✓ Comparison:        {comparison_path}")
print(f"  ✓ Summary:           {summary_path}")
print(f"  ✓ Paired predictions:{paired_predictions_path}")

print("=" * 84)

CELL 6 — MATCHED 49-FEATURE VS GPA_S1-EXCLUDED 48-FEATURE COMPARISON

Matched-design validation:
✓ Same 468 training students
✓ Same 117 independent-test students
✓ Same target labels
✓ Same locked hyperparameters
✓ Same seeds: [42, 123, 7]
✓ Same decision threshold: 0.50
✓ Only analytical difference = presence/absence of GPA_S1

------------------------------------------------------------------------------------
MATCHED 49-FEATURE REFERENCE — SEED 42
------------------------------------------------------------------------------------
ROC-AUC:     0.813817
PR-AUC:      0.775148
F1:          0.705882
Precision:   0.782609
Recall:      0.642857
Specificity: 0.836066
Accuracy:    0.743590
Confusion matrix: TN=51, FP=10, FN=20, TP=36

------------------------------------------------------------------------------------
MATCHED 49-FEATURE REFERENCE — SEED 123
------------------------------------------------------------------------------------
ROC-AUC:     0.813817
PR-AUC:      0.776971
F1:  

In [16]:
# ============================================================
# NOTEBOOK 16 — GPA_S1 EXCLUSION SENSITIVITY ANALYSIS
# CELL 7 — PAIRED STATISTICAL COMPARISON
#          MATCHED 49-FEATURE VS 48-FEATURE MODEL
#          PRIMARY INFERENCE SEED = 42
# ============================================================

import numpy as np
import pandas as pd

from scipy.stats import norm, binomtest

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix
)

print("=" * 86)
print("CELL 7 — PAIRED GPA_S1-EXCLUSION STATISTICAL SENSITIVITY ANALYSIS")
print("=" * 86)

# ------------------------------------------------------------
# 1. INFERENCE SETTINGS
# ------------------------------------------------------------
INFERENCE_SEED = 42
BOOTSTRAP_ITERATIONS = 5000
BOOTSTRAP_RANDOM_SEED = 42
ALPHA = 0.05

print("\nInference specification:")
print(f"  Matched seed:              {INFERENCE_SEED}")
print(f"  Independent-test students:{EXPECTED_TEST_N}")
print(f"  Bootstrap iterations:     {BOOTSTRAP_ITERATIONS}")
print(f"  Bootstrap RNG seed:       {BOOTSTRAP_RANDOM_SEED}")
print(f"  Significance level:       {ALPHA}")
print("  ROC comparison:           Paired DeLong")
print("  Metric uncertainty:       Paired stratified bootstrap")
print("  Classification comparison:Exact McNemar")
print("  Delta direction:          48-feature minus 49-feature")

# ------------------------------------------------------------
# 2. EXTRACT MATCHED SEED-42 PREDICTIONS
# ------------------------------------------------------------
paired_42 = (
    paired_predictions[
        paired_predictions["Seed"] == INFERENCE_SEED
    ]
    .sort_values("Test_Position")
    .reset_index(drop=True)
)

assert len(paired_42) == EXPECTED_TEST_N

y_true_42 = paired_42["y_true"].to_numpy(dtype=int)

prob_49 = paired_42["y_prob_49"].to_numpy(dtype=float)
pred_49 = paired_42["y_pred_49"].to_numpy(dtype=int)

prob_48 = paired_42[
    "y_prob_GPA_S1_excluded"
].to_numpy(dtype=float)

pred_48 = paired_42[
    "y_pred_GPA_S1_excluded"
].to_numpy(dtype=int)

assert np.array_equal(
    y_true_42,
    np.asarray(y_test, dtype=int)
)

print("\n✓ Seed-42 paired prediction alignment confirmed")

# ------------------------------------------------------------
# 3. RECONSTRUCT SEED-42 METRICS
# ------------------------------------------------------------
metrics_49, pred_49_check = calculate_binary_metrics(
    y_true=y_true_42,
    y_prob=prob_49,
    threshold=DECISION_THRESHOLD
)

metrics_48, pred_48_check = calculate_binary_metrics(
    y_true=y_true_42,
    y_prob=prob_48,
    threshold=DECISION_THRESHOLD
)

assert np.array_equal(pred_49, pred_49_check)
assert np.array_equal(pred_48, pred_48_check)

print("\nSeed-42 matched performance:")

for metric in metric_columns:

    delta = metrics_48[metric] - metrics_49[metric]

    print(
        f"  {metric:<12}: "
        f"49={metrics_49[metric]:.6f} | "
        f"48={metrics_48[metric]:.6f} | "
        f"Δ={delta:+.6f}"
    )

# ------------------------------------------------------------
# 4. DELONG INFLUENCE-VALUE IMPLEMENTATION
# ------------------------------------------------------------
def delong_auc_components(y_true, scores):
    """
    Compute AUC and DeLong influence components.

    Positive influence values:
        V10_i = mean_j phi(score_pos_i, score_neg_j)

    Negative influence values:
        V01_j = mean_i phi(score_pos_i, score_neg_j)
    """

    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    pos_scores = scores[y_true == 1]
    neg_scores = scores[y_true == 0]

    m = len(pos_scores)
    n = len(neg_scores)

    if m == 0 or n == 0:
        raise ValueError(
            "Both positive and negative observations "
            "are required for DeLong analysis."
        )

    comparison = (
        (pos_scores[:, None] > neg_scores[None, :]).astype(float)
        +
        0.5 * (
            pos_scores[:, None] == neg_scores[None, :]
        ).astype(float)
    )

    v10 = comparison.mean(axis=1)
    v01 = comparison.mean(axis=0)

    auc = comparison.mean()

    return auc, v10, v01


def paired_delong_test(
    y_true,
    scores_reference,
    scores_sensitivity
):
    """
    Paired DeLong test for difference in correlated ROC-AUCs.

    Difference direction:
        sensitivity - reference
    """

    auc_ref, v10_ref, v01_ref = delong_auc_components(
        y_true,
        scores_reference
    )

    auc_sens, v10_sens, v01_sens = delong_auc_components(
        y_true,
        scores_sensitivity
    )

    m = len(v10_ref)
    n = len(v01_ref)

    # Covariance among model influence functions
    cov_pos = np.cov(
        np.vstack([v10_ref, v10_sens]),
        ddof=1
    )

    cov_neg = np.cov(
        np.vstack([v01_ref, v01_sens]),
        ddof=1
    )

    auc_cov = (
        cov_pos / m
        +
        cov_neg / n
    )

    diff = auc_sens - auc_ref

    diff_var = (
        auc_cov[0, 0]
        + auc_cov[1, 1]
        - 2 * auc_cov[0, 1]
    )

    diff_var = max(float(diff_var), 0.0)
    se = np.sqrt(diff_var)

    if se > 0:
        z = diff / se
        p_value = 2 * norm.sf(abs(z))
    else:
        z = np.nan
        p_value = 1.0 if np.isclose(diff, 0) else 0.0

    ci_low = diff - 1.96 * se
    ci_high = diff + 1.96 * se

    return {
        "ROC_AUC_49": auc_ref,
        "ROC_AUC_48": auc_sens,
        "Delta_48_minus_49": diff,
        "Variance_Difference": diff_var,
        "SE_Difference": se,
        "Z": z,
        "P_Value": p_value,
        "CI95_Lower_Normal": ci_low,
        "CI95_Upper_Normal": ci_high
    }

# ------------------------------------------------------------
# 5. PAIRED DELONG TEST
# ------------------------------------------------------------
delong_result = paired_delong_test(
    y_true=y_true_42,
    scores_reference=prob_49,
    scores_sensitivity=prob_48
)

print("\n" + "=" * 86)
print("PAIRED DELONG ROC-AUC COMPARISON")
print("=" * 86)

for key, value in delong_result.items():
    if isinstance(value, (float, np.floating)):
        print(f"{key:<26}: {value:.8f}")
    else:
        print(f"{key:<26}: {value}")

# ------------------------------------------------------------
# 6. BOOTSTRAP METRIC HELPER
# ------------------------------------------------------------
def metric_vector(y_true, y_prob, threshold=0.50):

    y_pred = (
        np.asarray(y_prob) >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    return {
        "ROC_AUC": roc_auc_score(
            y_true,
            y_prob
        ),
        "PR_AUC": average_precision_score(
            y_true,
            y_prob
        ),
        "F1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Specificity": specificity,
        "Accuracy": accuracy_score(
            y_true,
            y_pred
        )
    }

# ------------------------------------------------------------
# 7. PAIRED STRATIFIED BOOTSTRAP
# ------------------------------------------------------------
positive_idx = np.where(
    y_true_42 == 1
)[0]

negative_idx = np.where(
    y_true_42 == 0
)[0]

assert len(positive_idx) == 56
assert len(negative_idx) == 61

rng = np.random.default_rng(
    BOOTSTRAP_RANDOM_SEED
)

bootstrap_deltas = {
    metric: []
    for metric in metric_columns
}

for b in range(BOOTSTRAP_ITERATIONS):

    sampled_positive = rng.choice(
        positive_idx,
        size=len(positive_idx),
        replace=True
    )

    sampled_negative = rng.choice(
        negative_idx,
        size=len(negative_idx),
        replace=True
    )

    boot_idx = np.concatenate([
        sampled_negative,
        sampled_positive
    ])

    y_b = y_true_42[boot_idx]

    prob49_b = prob_49[boot_idx]
    prob48_b = prob_48[boot_idx]

    metrics49_b = metric_vector(
        y_b,
        prob49_b,
        DECISION_THRESHOLD
    )

    metrics48_b = metric_vector(
        y_b,
        prob48_b,
        DECISION_THRESHOLD
    )

    for metric in metric_columns:

        bootstrap_deltas[metric].append(
            metrics48_b[metric]
            -
            metrics49_b[metric]
        )

# ------------------------------------------------------------
# 8. BOOTSTRAP CONFIDENCE INTERVALS
# ------------------------------------------------------------
bootstrap_rows = []

observed_deltas = {
    metric:
        metrics_48[metric]
        -
        metrics_49[metric]
    for metric in metric_columns
}

for metric in metric_columns:

    values = np.asarray(
        bootstrap_deltas[metric],
        dtype=float
    )

    lower = np.percentile(
        values,
        100 * (ALPHA / 2)
    )

    upper = np.percentile(
        values,
        100 * (1 - ALPHA / 2)
    )

    bootstrap_rows.append({
        "Metric": metric,
        "Reference_49": metrics_49[metric],
        "GPA_S1_Excluded_48": metrics_48[metric],
        "Observed_Delta_48_minus_49":
            observed_deltas[metric],
        "Bootstrap_Mean_Delta":
            np.mean(values),
        "Bootstrap_SD":
            np.std(values, ddof=1),
        "CI95_Lower":
            lower,
        "CI95_Upper":
            upper,
        "CI_Excludes_Zero":
            bool(
                (lower > 0)
                or
                (upper < 0)
            )
    })

bootstrap_results = pd.DataFrame(
    bootstrap_rows
)

print("\n" + "=" * 86)
print("PAIRED STRATIFIED BOOTSTRAP — 95% CONFIDENCE INTERVALS")
print("=" * 86)

print(
    bootstrap_results[
        [
            "Metric",
            "Reference_49",
            "GPA_S1_Excluded_48",
            "Observed_Delta_48_minus_49",
            "CI95_Lower",
            "CI95_Upper",
            "CI_Excludes_Zero"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ------------------------------------------------------------
# 9. EXACT MCNEMAR TEST
# ------------------------------------------------------------
correct_49 = (
    pred_49 == y_true_42
)

correct_48 = (
    pred_48 == y_true_42
)

both_correct = int(
    np.sum(correct_49 & correct_48)
)

only_49_correct = int(
    np.sum(correct_49 & ~correct_48)
)

only_48_correct = int(
    np.sum(~correct_49 & correct_48)
)

both_wrong = int(
    np.sum(~correct_49 & ~correct_48)
)

discordant_total = (
    only_49_correct
    +
    only_48_correct
)

if discordant_total > 0:

    mcnemar_result = binomtest(
        k=min(
            only_49_correct,
            only_48_correct
        ),
        n=discordant_total,
        p=0.5,
        alternative="two-sided"
    )

    mcnemar_p = mcnemar_result.pvalue

else:

    mcnemar_p = 1.0

print("\n" + "=" * 86)
print("EXACT MCNEMAR CLASSIFICATION COMPARISON")
print("=" * 86)

print(f"Both correct:       {both_correct}")
print(f"49-feature only:    {only_49_correct}")
print(f"48-feature only:    {only_48_correct}")
print(f"Both incorrect:     {both_wrong}")
print(f"Discordant pairs:   {discordant_total}")
print(f"Exact McNemar p:    {mcnemar_p:.8f}")

# ------------------------------------------------------------
# 10. MCNEMAR TABLE
# ------------------------------------------------------------
mcnemar_table = pd.DataFrame({
    "Comparison": [
        "Both models correct",
        "49-feature correct / 48-feature wrong",
        "49-feature wrong / 48-feature correct",
        "Both models wrong"
    ],
    "Count": [
        both_correct,
        only_49_correct,
        only_48_correct,
        both_wrong
    ]
})

# ------------------------------------------------------------
# 11. SAVE STATISTICAL RESULTS
# ------------------------------------------------------------
delong_path = (
    SENSITIVITY_DIR /
    "gpa_s1_exclusion_seed42_paired_delong.csv"
)

bootstrap_path = (
    SENSITIVITY_DIR /
    "gpa_s1_exclusion_seed42_paired_bootstrap.csv"
)

mcnemar_path = (
    SENSITIVITY_DIR /
    "gpa_s1_exclusion_seed42_exact_mcnemar.csv"
)

delong_df = pd.DataFrame([
    delong_result
])

delong_df.to_csv(
    delong_path,
    index=False
)

bootstrap_results.to_csv(
    bootstrap_path,
    index=False
)

mcnemar_export = mcnemar_table.copy()

mcnemar_export[
    "Exact_McNemar_P"
] = np.nan

mcnemar_export.loc[
    0,
    "Exact_McNemar_P"
] = mcnemar_p

mcnemar_export.to_csv(
    mcnemar_path,
    index=False
)

# ------------------------------------------------------------
# 12. FINAL VALIDATION
# ------------------------------------------------------------
assert len(paired_42) == 117
assert len(positive_idx) == 56
assert len(negative_idx) == 61

assert len(
    bootstrap_results
) == len(metric_columns)

print("\n" + "=" * 86)
print("CELL 7 FINAL VALIDATION")
print("=" * 86)

print("✓ Same 117 students used for both models")
print("✓ Seed-42 matched inference performed")
print("✓ Paired DeLong ROC-AUC comparison completed")
print(
    f"✓ {BOOTSTRAP_ITERATIONS} paired stratified "
    "bootstrap replicates completed"
)
print("✓ Positive/negative class counts preserved during bootstrap")
print("✓ Exact McNemar comparison completed")
print("✓ No three-seed pseudo-replication used")
print("✓ No model tuning or feature selection performed")
print("✓ Delta convention = 48-feature minus 49-feature")
print("✓ Primary locked SP-XGBoost results remain unchanged")

print("\nSaved artifacts:")
print(f"  ✓ DeLong:    {delong_path}")
print(f"  ✓ Bootstrap: {bootstrap_path}")
print(f"  ✓ McNemar:   {mcnemar_path}")

print("=" * 86)

CELL 7 — PAIRED GPA_S1-EXCLUSION STATISTICAL SENSITIVITY ANALYSIS

Inference specification:
  Matched seed:              42
  Independent-test students:117
  Bootstrap iterations:     5000
  Bootstrap RNG seed:       42
  Significance level:       0.05
  ROC comparison:           Paired DeLong
  Metric uncertainty:       Paired stratified bootstrap
  Classification comparison:Exact McNemar
  Delta direction:          48-feature minus 49-feature

✓ Seed-42 paired prediction alignment confirmed

Seed-42 matched performance:
  ROC_AUC     : 49=0.813817 | 48=0.523126 | Δ=-0.290691
  PR_AUC      : 49=0.775148 | 48=0.550638 | Δ=-0.224509
  F1          : 49=0.705882 | 48=0.523364 | Δ=-0.182518
  Precision   : 49=0.782609 | 48=0.549020 | Δ=-0.233589
  Recall      : 49=0.642857 | 48=0.500000 | Δ=-0.142857
  Specificity : 49=0.836066 | 48=0.622951 | Δ=-0.213115
  Accuracy    : 49=0.743590 | 48=0.564103 | Δ=-0.179487

PAIRED DELONG ROC-AUC COMPARISON
ROC_AUC_49                : 0.81381733
ROC_AUC

In [17]:
# ============================================================
# NOTEBOOK 16 — GPA_S1 EXCLUSION SENSITIVITY ANALYSIS
# CELL 8 — FINAL CONSOLIDATION, VALIDATION,
#          AND MANUSCRIPT-READY SUMMARY
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

print("=" * 90)
print("CELL 8 — FINAL GPA_S1 EXCLUSION SENSITIVITY CONSOLIDATION")
print("=" * 90)

# ------------------------------------------------------------
# 1. CORE ANALYSIS METADATA
# ------------------------------------------------------------
analysis_metadata = {
    "analysis_name": "GPA_S1 Exclusion Sensitivity Analysis",
    "analysis_type": "Post-hoc supplementary sensitivity analysis",
    "primary_pipeline_status": "LOCKED — unchanged",
    "excluded_feature": EXCLUDED_FEATURE,
    "locked_feature_count": 49,
    "sensitivity_feature_count": 48,
    "train_n": EXPECTED_TRAIN_N,
    "test_n": EXPECTED_TEST_N,
    "seeds": SEEDS,
    "primary_inference_seed": INFERENCE_SEED,
    "decision_threshold": DECISION_THRESHOLD,
    "feature_reselection_performed": False,
    "hyperparameter_retuning_performed": False,
    "test_set_used_for_tuning": False,
    "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
    "bootstrap_seed": BOOTSTRAP_RANDOM_SEED,
    "alpha": ALPHA
}

# ------------------------------------------------------------
# 2. THREE-SEED PERFORMANCE SUMMARY
# ------------------------------------------------------------
three_seed_summary_rows = []

for metric in metric_columns:

    row = matched_summary[
        matched_summary["Metric"] == metric
    ].iloc[0]

    three_seed_summary_rows.append({
        "Metric": metric,
        "49_Feature_Mean": row["Reference_49_Mean"],
        "49_Feature_SD": row["Reference_49_SD"],
        "48_Feature_Mean": row["GPA_S1_Excluded_48_Mean"],
        "48_Feature_SD": row["GPA_S1_Excluded_48_SD"],
        "Mean_Delta_48_minus_49": row["Mean_Delta_48_minus_49"],
        "Mean_Percent_Change": row["Mean_Percent_Change"]
    })

final_three_seed_summary = pd.DataFrame(
    three_seed_summary_rows
)

# ------------------------------------------------------------
# 3. SEED-42 PAIRED INFERENCE SUMMARY
# ------------------------------------------------------------
delong_p = float(
    delong_result["P_Value"]
)

delong_ci_low = float(
    delong_result["CI95_Lower_Normal"]
)

delong_ci_high = float(
    delong_result["CI95_Upper_Normal"]
)

paired_inference_rows = []

for metric in metric_columns:

    boot_row = bootstrap_results[
        bootstrap_results["Metric"] == metric
    ].iloc[0]

    paired_inference_rows.append({
        "Metric": metric,
        "49_Feature_Seed42": boot_row["Reference_49"],
        "48_Feature_Seed42": boot_row["GPA_S1_Excluded_48"],
        "Delta_48_minus_49": boot_row[
            "Observed_Delta_48_minus_49"
        ],
        "Bootstrap_CI95_Lower": boot_row["CI95_Lower"],
        "Bootstrap_CI95_Upper": boot_row["CI95_Upper"],
        "Bootstrap_CI_Excludes_Zero": boot_row[
            "CI_Excludes_Zero"
        ]
    })

final_paired_inference = pd.DataFrame(
    paired_inference_rows
)

# Add ROC-specific DeLong inference
final_paired_inference[
    "DeLong_P_Value"
] = np.nan

final_paired_inference[
    "DeLong_CI95_Lower"
] = np.nan

final_paired_inference[
    "DeLong_CI95_Upper"
] = np.nan

roc_index = final_paired_inference[
    "Metric"
] == "ROC_AUC"

final_paired_inference.loc[
    roc_index,
    "DeLong_P_Value"
] = delong_p

final_paired_inference.loc[
    roc_index,
    "DeLong_CI95_Lower"
] = delong_ci_low

final_paired_inference.loc[
    roc_index,
    "DeLong_CI95_Upper"
] = delong_ci_high

# ------------------------------------------------------------
# 4. MCNEMAR SUMMARY
# ------------------------------------------------------------
final_mcnemar_summary = pd.DataFrame({
    "Both_Correct": [both_correct],
    "49_Feature_Only_Correct": [only_49_correct],
    "48_Feature_Only_Correct": [only_48_correct],
    "Both_Incorrect": [both_wrong],
    "Discordant_Pairs": [discordant_total],
    "Exact_McNemar_P": [mcnemar_p]
})

# ------------------------------------------------------------
# 5. FINAL VALIDATION RULES
# ------------------------------------------------------------
validation_checks = {
    "locked_primary_pipeline_unchanged":
        ANALYSIS_RULES["primary_pipeline_modified"] is False,

    "exactly_one_feature_removed":
        len(locked_features) - len(sensitivity_features) == 1,

    "removed_feature_is_GPA_S1":
        EXCLUDED_FEATURE not in sensitivity_features,

    "train_sample_preserved":
        len(X_train_48) == EXPECTED_TRAIN_N,

    "test_sample_preserved":
        len(X_test_48) == EXPECTED_TEST_N,

    "same_target_labels":
        np.array_equal(
            np.asarray(y_test),
            y_true_42
        ),

    "same_locked_hyperparameters":
        params_validation_df["match"].all(),

    "no_reselection":
        ANALYSIS_RULES[
            "feature_reselection_permitted"
        ] is False,

    "no_retuning":
        ANALYSIS_RULES[
            "hyperparameter_retuning_permitted"
        ] is False,

    "paired_predictions_available":
        len(paired_predictions)
        == EXPECTED_TEST_N * len(SEEDS),

    "paired_seed42_n_117":
        len(paired_42) == EXPECTED_TEST_N,

    "bootstrap_5000_completed":
        BOOTSTRAP_ITERATIONS == 5000
}

assert all(validation_checks.values()), (
    "One or more final validation checks failed."
)

# ------------------------------------------------------------
# 6. CREATE REVIEWER-READY RESULTS TABLE
# ------------------------------------------------------------
reviewer_table = final_three_seed_summary.copy()

for col in [
    "49_Feature_Mean",
    "49_Feature_SD",
    "48_Feature_Mean",
    "48_Feature_SD",
    "Mean_Delta_48_minus_49",
    "Mean_Percent_Change"
]:
    reviewer_table[col] = reviewer_table[col].astype(float)

print("\n" + "=" * 90)
print("REVIEWER-READY THREE-SEED SENSITIVITY SUMMARY")
print("=" * 90)

print(
    reviewer_table.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# ------------------------------------------------------------
# 7. DISPLAY PAIRED INFERENCE SUMMARY
# ------------------------------------------------------------
print("\n" + "=" * 90)
print("SEED-42 PAIRED INFERENCE SUMMARY")
print("=" * 90)

print(
    final_paired_inference.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nExact McNemar:")
print(f"  49-feature only correct: {only_49_correct}")
print(f"  48-feature only correct: {only_48_correct}")
print(f"  Exact p-value:           {mcnemar_p:.8f}")

# ------------------------------------------------------------
# 8. MANUSCRIPT-READY METHODS TEXT
# ------------------------------------------------------------
methods_text = (
    "As a supplementary post-hoc sensitivity analysis, GPA_S1, "
    "the highest-ranked predictor in the locked SHAP-selected "
    "feature space, was excluded from the final SP-XGBoost model. "
    "The sensitivity analysis was implemented on a separate cloned "
    "branch and did not alter the primary pipeline or reported primary "
    "results. The remaining 48 predictors were evaluated using the "
    "same 468/117 stratified training-test partition, locked "
    "SP-XGBoost hyperparameters, decision threshold of 0.50, and "
    "random seeds (42, 123, and 7). No additional feature selection "
    "or hyperparameter optimisation was performed, thereby isolating "
    "the effect of removing GPA_S1. Performance was summarised across "
    "the three seeds. For paired inference on the same 117 independent "
    "test observations, seed 42 was used for a paired DeLong comparison "
    "of ROC-AUC, 5,000 paired stratified bootstrap replicates for "
    "confidence intervals of metric differences, and an exact McNemar "
    "test for differences in thresholded classification correctness."
)

# ------------------------------------------------------------
# 9. MANUSCRIPT-READY RESULTS TEXT
# ------------------------------------------------------------
roc_summary = final_three_seed_summary[
    final_three_seed_summary["Metric"] == "ROC_AUC"
].iloc[0]

results_text = (
    "Excluding GPA_S1 produced a marked reduction in predictive "
    "performance. Across the three matched random seeds, mean ROC-AUC "
    f"decreased from {roc_summary['49_Feature_Mean']:.4f} "
    f"± {roc_summary['49_Feature_SD']:.4f} with the locked 49-feature "
    f"model to {roc_summary['48_Feature_Mean']:.4f} "
    f"± {roc_summary['48_Feature_SD']:.4f} after GPA_S1 exclusion, "
    f"corresponding to a mean absolute change of "
    f"{roc_summary['Mean_Delta_48_minus_49']:.4f}. "
    "In the matched seed-42 paired analysis, ROC-AUC decreased from "
    f"{delong_result['ROC_AUC_49']:.4f} to "
    f"{delong_result['ROC_AUC_48']:.4f} "
    f"(Δ = {delong_result['Delta_48_minus_49']:.4f}, "
    f"paired DeLong p = {delong_result['P_Value']:.2e}; "
    f"95% CI {delong_result['CI95_Lower_Normal']:.4f} to "
    f"{delong_result['CI95_Upper_Normal']:.4f}). "
    "Paired bootstrap confidence intervals also excluded zero for "
    "ROC-AUC, PR-AUC, F1-score, precision, specificity, and accuracy, "
    "whereas the recall interval included zero. At the 0.50 decision "
    "threshold, the 49-feature model correctly classified 32 students "
    "that the GPA_S1-excluded model misclassified, compared with "
    "11 students correctly classified only by the GPA_S1-excluded "
    f"model (exact McNemar p = {mcnemar_p:.4f}). "
    "These findings indicate that the model's discrimination was "
    "strongly dependent on predictive information carried by GPA_S1."
)

# ------------------------------------------------------------
# 10. INTERPRETATION BOUNDARY
# ------------------------------------------------------------
interpretation_boundary = (
    "The GPA_S1-exclusion analysis is supplementary and should not be "
    "interpreted as evidence that GPA_S1 causally determines later "
    "academic risk. It evaluates the dependence of the fitted model's "
    "predictive performance on information contained in GPA_S1. "
    "The sensitivity analysis does not replace the locked primary "
    "SP-XGBoost model, its original independent-test performance, "
    "or the previously reported robustness analysis."
)

# ------------------------------------------------------------
# 11. SAVE TEXT OUTPUTS
# ------------------------------------------------------------
text_summary_path = (
    SENSITIVITY_DIR /
    "gpa_s1_exclusion_manuscript_ready_text.txt"
)

with open(
    text_summary_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "METHODS\n"
        "=======\n"
        f"{methods_text}\n\n"
    )

    f.write(
        "RESULTS\n"
        "=======\n"
        f"{results_text}\n\n"
    )

    f.write(
        "INTERPRETATION BOUNDARY\n"
        "=======================\n"
        f"{interpretation_boundary}\n"
    )

# ------------------------------------------------------------
# 12. SAVE FINAL TABLES
# ------------------------------------------------------------
final_three_seed_path = (
    SENSITIVITY_DIR /
    "final_gpa_s1_exclusion_three_seed_summary.csv"
)

final_inference_path = (
    SENSITIVITY_DIR /
    "final_gpa_s1_exclusion_paired_inference.csv"
)

final_mcnemar_path = (
    SENSITIVITY_DIR /
    "final_gpa_s1_exclusion_mcnemar_summary.csv"
)

reviewer_table.to_csv(
    final_three_seed_path,
    index=False
)

final_paired_inference.to_csv(
    final_inference_path,
    index=False
)

final_mcnemar_summary.to_csv(
    final_mcnemar_path,
    index=False
)

# ------------------------------------------------------------
# 13. SAVE FINAL ANALYSIS MANIFEST
# ------------------------------------------------------------
final_manifest = {
    **analysis_metadata,
    "validation_checks": {
        key: bool(value)
        for key, value
        in validation_checks.items()
    },
    "key_result": {
        "three_seed_reference_roc_auc_mean":
            float(
                roc_summary["49_Feature_Mean"]
            ),
        "three_seed_excluded_roc_auc_mean":
            float(
                roc_summary[
                    "48_Feature_Mean"
                ]
            ),
        "three_seed_mean_roc_auc_delta":
            float(
                roc_summary[
                    "Mean_Delta_48_minus_49"
                ]
            ),
        "seed42_delong_p":
            float(delong_p),
        "seed42_mcnemar_p":
            float(mcnemar_p)
    }
}

manifest_path = (
    SENSITIVITY_DIR /
    "notebook16_final_analysis_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        final_manifest,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 14. FINAL COMPLETION REPORT
# ------------------------------------------------------------
print("\n" + "=" * 90)
print("NOTEBOOK 16 — FINAL VALIDATION")
print("=" * 90)

for check, passed in validation_checks.items():
    symbol = "✓" if passed else "✗"
    print(f"{symbol} {check}")

print("\nPrimary sensitivity finding:")
print(
    f"  49-feature ROC-AUC: "
    f"{roc_summary['49_Feature_Mean']:.4f} ± "
    f"{roc_summary['49_Feature_SD']:.4f}"
)
print(
    f"  48-feature ROC-AUC: "
    f"{roc_summary['48_Feature_Mean']:.4f} ± "
    f"{roc_summary['48_Feature_SD']:.4f}"
)
print(
    f"  Mean ΔROC-AUC: "
    f"{roc_summary['Mean_Delta_48_minus_49']:+.4f}"
)
print(
    f"  Seed-42 DeLong p: "
    f"{delong_p:.8f}"
)
print(
    f"  Seed-42 McNemar p: "
    f"{mcnemar_p:.8f}"
)

print("\nSaved final artifacts:")
print(f"  ✓ Three-seed summary: {final_three_seed_path}")
print(f"  ✓ Paired inference:   {final_inference_path}")
print(f"  ✓ McNemar summary:    {final_mcnemar_path}")
print(f"  ✓ Manuscript text:    {text_summary_path}")
print(f"  ✓ Final manifest:     {manifest_path}")

print("\n" + "=" * 90)
print("NOTEBOOK 16 COMPLETE")
print("=" * 90)

print(
    "✓ GPA_S1 exclusion sensitivity analysis completed"
)
print(
    "✓ Primary SP-XGBoost pipeline remains locked and unchanged"
)
print(
    "✓ Sensitivity results are supplementary and separately stored"
)
print(
    "✓ Notebook 16 may now be closed after artifact verification"
)
print("=" * 90)

CELL 8 — FINAL GPA_S1 EXCLUSION SENSITIVITY CONSOLIDATION

REVIEWER-READY THREE-SEED SENSITIVITY SUMMARY
     Metric  49_Feature_Mean  49_Feature_SD  48_Feature_Mean  48_Feature_SD  Mean_Delta_48_minus_49  Mean_Percent_Change
    ROC_AUC           0.8132         0.0010           0.5281         0.0065                 -0.2851             -35.0605
     PR_AUC           0.7755         0.0014           0.5607         0.0186                 -0.2147             -27.6880
         F1           0.6996         0.0173           0.5108         0.0118                 -0.1888             -26.9710
  Precision           0.7851         0.0138           0.5289         0.0174                 -0.2562             -32.6127
     Recall           0.6310         0.0206           0.4940         0.0103                 -0.1369             -21.6776
Specificity           0.8415         0.0095           0.5956         0.0250                 -0.2459             -29.1981
   Accuracy           0.7407         0.0131     